# Helm and GitOps: Production Kubernetes Deployment

Kubernetes gives you the power to run containerized workloads at scale. But raw Kubernetes manifests quickly become unmanageable: dozens of YAML files, repeated values across files, no versioning of deployments, and manual `kubectl apply` commands that are hard to audit or roll back.

This notebook covers two tools that solve this:

- **Helm**: the package manager for Kubernetes, turning repeated YAML into parameterized templates
- **GitOps**: a deployment approach where Git is the single source of truth, and a cluster operator continuously reconciles the cluster to match the declared state in Git

Together they form the backbone of production ML deployment pipelines.

## Why Helm?

Imagine deploying an ML inference service. You need at minimum:
- A Deployment (manages pods)
- A Service (exposes the deployment)
- A HorizontalPodAutoscaler (scales replicas by CPU/custom metrics)
- An Ingress (routes external traffic)
- ConfigMaps and Secrets (environment variables, model config)

Now deploy this to three environments: dev, staging, prod. Each has different image tags, replica counts, resource limits, and domain names. With raw YAML you copy-paste everything and update values by hand. One missed sed replace creates a prod outage.

Helm solves this with templates and values. You define the structure once, parameterize every environment-specific value, and install with a single command.

In [1]:
# Helm uses the same concept as Python's Jinja2 templates
# but with Go template syntax {{ .Values.key }}

# Think of it like this Python analogy:
import json

values = {
    'image': {'repository': 'myregistry/ml-model', 'tag': 'v1.2.0'},
    'replicaCount': 3,
    'resources': {'limits': {'cpu': '2', 'memory': '4Gi'}, 'requests': {'cpu': '500m', 'memory': '1Gi'}},
    'env': {'MODEL_NAME': 'fraud-detection', 'LOG_LEVEL': 'INFO'},
    'autoscaling': {'enabled': True, 'minReplicas': 2, 'maxReplicas': 10, 'targetCPUUtilizationPercentage': 70}
}

# In Helm, a template file accesses these like:
# image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
# replicas: {{ .Values.replicaCount }}

# Python equivalent for illustration:
template = """
image: {repo}:{tag}
replicas: {replicas}
cpu_limit: {cpu}
""".format(
    repo=values['image']['repository'],
    tag=values['image']['tag'],
    replicas=values['replicaCount'],
    cpu=values['resources']['limits']['cpu']
)

print('Rendered from values:')
print(template)
print('This is exactly what Helm does, but for entire Kubernetes manifests.')

Rendered from values:

image: myregistry/ml-model:v1.2.0
replicas: 3
cpu_limit: 2

This is exactly what Helm does, but for entire Kubernetes manifests.


## Section 1: Helm Concepts

Helm has four core concepts:

| Concept | Description | Analogy |
|---|---|---|
| **Chart** | A package of Kubernetes manifests with templates | Python package (wheel/sdist) |
| **Values** | Customizable parameters for a chart | Config file / environment variables |
| **Release** | An installed instance of a chart in a cluster | A running Python process |
| **Repository** | Where charts are stored and shared | PyPI |

You can install the same chart multiple times (dev, staging, prod) creating three separate releases, each with different values. Helm tracks release history so you can roll back to a previous revision.

Popular chart repositories:
- **ArtifactHub** (https://artifacthub.io): the main public registry
- **Bitnami**: production-grade charts for common software
- **Company-internal registries**: OCI-compatible (push to container registry)

## Section 2: Helm Chart Structure

In [2]:
# Helm chart directory structure
chart_structure = {
    'ml-model-chart/': {
        'Chart.yaml': 'Chart metadata: name, version, description, appVersion',
        'values.yaml': 'Default values for template parameters',
        'values-dev.yaml': 'Dev environment overrides (optional)',
        'values-prod.yaml': 'Prod environment overrides (optional)',
        '.helmignore': 'Files to exclude when packaging (like .gitignore)',
        'templates/': {
            '_helpers.tpl': 'Reusable template fragments (named templates)',
            'deployment.yaml': 'Kubernetes Deployment manifest (templated)',
            'service.yaml': 'Kubernetes Service manifest (templated)',
            'ingress.yaml': 'Ingress for external traffic routing',
            'hpa.yaml': 'HorizontalPodAutoscaler for auto-scaling',
            'configmap.yaml': 'ConfigMap for model configuration',
            'NOTES.txt': 'Post-install instructions printed to user'
        },
        'charts/': 'Sub-charts (dependencies, e.g., Redis, PostgreSQL)'
    }
}

def print_tree(d, indent=0):
    for key, value in d.items():
        if isinstance(value, dict):
            print(' ' * indent + f'|-- {key}')
            print_tree(value, indent + 4)
        else:
            print(' ' * indent + f'|-- {key}')
            print(' ' * (indent + 4) + f'# {value}')

print_tree(chart_structure)

|-- ml-model-chart/
    |-- Chart.yaml
        # Chart metadata: name, version, description, appVersion
    |-- values.yaml
        # Default values for template parameters
    |-- values-dev.yaml
        # Dev environment overrides (optional)
    |-- values-prod.yaml
        # Prod environment overrides (optional)
    |-- .helmignore
        # Files to exclude when packaging (like .gitignore)
    |-- templates/
        |-- _helpers.tpl
            # Reusable template fragments (named templates)
        |-- deployment.yaml
            # Kubernetes Deployment manifest (templated)
        |-- service.yaml
            # Kubernetes Service manifest (templated)
        |-- ingress.yaml
            # Ingress for external traffic routing
        |-- hpa.yaml
            # HorizontalPodAutoscaler for auto-scaling
        |-- configmap.yaml
            # ConfigMap for model configuration
        |-- NOTES.txt
            # Post-install instructions printed to user
    |-- charts/
        # Sub-

In [3]:
# Chart.yaml
chart_yaml = """
apiVersion: v2
name: ml-model-chart
description: Helm chart for deploying ML inference services
type: application
version: 0.4.1          # Chart version (semver). Increment when chart changes.
appVersion: "2.1.0"     # Version of the app being packaged (informational)
keywords:
  - machine-learning
  - inference
  - api
maintainers:
  - name: ML Platform Team
    email: mlplatform@company.com
"""

print('Chart.yaml:')
print(chart_yaml)

Chart.yaml:

apiVersion: v2
name: ml-model-chart
description: Helm chart for deploying ML inference services
type: application
version: 0.4.1          # Chart version (semver). Increment when chart changes.
appVersion: "2.1.0"     # Version of the app being packaged (informational)
keywords:
  - machine-learning
  - inference
  - api
maintainers:
  - name: ML Platform Team
    email: mlplatform@company.com



In [4]:
# values.yaml: the default configuration for the chart
# Teams override specific values per environment without changing this file
values_yaml = """
# Default values for ml-model-chart.
# Override with: helm install my-release ./chart --values values-prod.yaml
# Or inline:     helm install my-release ./chart --set image.tag=v2.0

replicaCount: 2

image:
  repository: myregistry.azurecr.io/fraud-detection
  pullPolicy: IfNotPresent
  tag: "latest"         # Override this in CI/CD: --set image.tag=$GIT_SHA

imagePullSecrets:
  - name: registry-credentials

service:
  type: ClusterIP
  port: 8080

ingress:
  enabled: true
  className: nginx
  host: ml-api.company.com
  tlsSecretName: ml-api-tls

resources:
  requests:
    cpu: "500m"          # 0.5 CPU cores requested
    memory: "1Gi"
  limits:
    cpu: "2"
    memory: "4Gi"        # OOM kill if exceeded

autoscaling:
  enabled: true
  minReplicas: 2
  maxReplicas: 20
  targetCPUUtilizationPercentage: 70

env:
  MODEL_NAME: fraud-detection-v2
  LOG_LEVEL: INFO
  MAX_BATCH_SIZE: "32"
  ONNX_THREADS: "4"

livenessProbe:
  httpGet:
    path: /health
    port: 8080
  initialDelaySeconds: 30
  periodSeconds: 10

readinessProbe:
  httpGet:
    path: /ready
    port: 8080
  initialDelaySeconds: 15
  periodSeconds: 5
"""

print('values.yaml:')
print(values_yaml)

values.yaml:

# Default values for ml-model-chart.
# Override with: helm install my-release ./chart --values values-prod.yaml
# Or inline:     helm install my-release ./chart --set image.tag=v2.0

replicaCount: 2

image:
  repository: myregistry.azurecr.io/fraud-detection
  pullPolicy: IfNotPresent
  tag: "latest"         # Override this in CI/CD: --set image.tag=$GIT_SHA

imagePullSecrets:
  - name: registry-credentials

service:
  type: ClusterIP
  port: 8080

ingress:
  enabled: true
  className: nginx
  host: ml-api.company.com
  tlsSecretName: ml-api-tls

resources:
  requests:
    cpu: "500m"          # 0.5 CPU cores requested
    memory: "1Gi"
  limits:
    cpu: "2"
    memory: "4Gi"        # OOM kill if exceeded

autoscaling:
  enabled: true
  minReplicas: 2
  maxReplicas: 20
  targetCPUUtilizationPercentage: 70

env:
  MODEL_NAME: fraud-detection-v2
  LOG_LEVEL: INFO
  MAX_BATCH_SIZE: "32"
  ONNX_THREADS: "4"

livenessProbe:
  httpGet:
    path: /health
    port: 8080
  initia

In [5]:
# templates/deployment.yaml
# {{ .Values.xxx }} is replaced at install/upgrade time with values from values.yaml
# {{ .Release.Name }} is the name given at helm install time
# {{ include "ml-model-chart.fullname" . }} calls a helper from _helpers.tpl
deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include \"ml-model-chart.fullname\" . }}
  labels:
    app: {{ .Release.Name }}
    chart: {{ .Chart.Name }}-{{ .Chart.Version }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}
    spec:
      imagePullSecrets:
        {{- toYaml .Values.imagePullSecrets | nindent 8 }}
      containers:
        - name: model-server
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          imagePullPolicy: {{ .Values.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.service.port }}
          env:
            {{- range $key, $val := .Values.env }}
            - name: {{ $key }}
              value: {{ $val | quote }}
            {{- end }}
          resources:
            {{- toYaml .Values.resources | nindent 12 }}
          livenessProbe:
            {{- toYaml .Values.livenessProbe | nindent 12 }}
          readinessProbe:
            {{- toYaml .Values.readinessProbe | nindent 12 }}
"""

print('templates/deployment.yaml (with Go template syntax):')
print(deployment_yaml)

templates/deployment.yaml (with Go template syntax):

apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "ml-model-chart.fullname" . }}
  labels:
    app: {{ .Release.Name }}
    chart: {{ .Chart.Name }}-{{ .Chart.Version }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}
    spec:
      imagePullSecrets:
        {{- toYaml .Values.imagePullSecrets | nindent 8 }}
      containers:
        - name: model-server
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          imagePullPolicy: {{ .Values.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.service.port }}
          env:
            {{- range $key, $val := .Values.env }}
            - name: {{ $key }}
              value: {{ $val | quote }}
            {{- end }}
          resources:
            {{- toYaml .Values.resources | nindent 12 }}
 

## Section 3: Helm Commands

The most important Helm commands cover the full release lifecycle.

In [6]:
helm_commands = [
    (
        'Install (first deployment)',
        'helm install my-model ./ml-model-chart --namespace ml-prod --create-namespace --set image.tag=v1.2.0',
        'Creates a new release named my-model. Fails if release already exists.'
    ),
    (
        'Upgrade (new version)',
        'helm upgrade my-model ./ml-model-chart --namespace ml-prod --set image.tag=v1.3.0',
        'Updates the existing release to new values. Kubernetes does rolling update of pods.'
    ),
    (
        'Install or upgrade (idempotent)',
        'helm upgrade --install my-model ./ml-model-chart --namespace ml-prod --set image.tag=v1.3.0',
        'Safe to run repeatedly. Used in CI/CD: installs if not present, upgrades if present.'
    ),
    (
        'Upgrade with values file',
        'helm upgrade my-model ./ml-model-chart -f values-prod.yaml --set image.tag=v1.3.0',
        'Use a values file for stable config, --set for per-deploy overrides like image tag.'
    ),
    (
        'Rollback to previous revision',
        'helm rollback my-model 1',
        'Rolls back to revision 1. Helm keeps full release history. Revision 0 = previous.'
    ),
    (
        'View release history',
        'helm history my-model --namespace ml-prod',
        'Shows all revisions: revision number, timestamp, status, chart version, description.'
    ),
    (
        'Preview without deploying',
        'helm template my-model ./ml-model-chart -f values-prod.yaml | kubectl diff -f -',
        'Renders templates locally and diffs against cluster state. No changes applied.'
    ),
    (
        'List all releases',
        'helm list --all-namespaces',
        'Shows all installed releases across all namespaces.'
    ),
    (
        'Uninstall',
        'helm uninstall my-model --namespace ml-prod',
        'Deletes all Kubernetes resources created by the release.'
    ),
]

for label, cmd, explanation in helm_commands:
    print(f'[{label}]')
    print(f'  $ {cmd}')
    print(f'  -> {explanation}')
    print()

[Install (first deployment)]
  $ helm install my-model ./ml-model-chart --namespace ml-prod --create-namespace --set image.tag=v1.2.0
  -> Creates a new release named my-model. Fails if release already exists.

[Upgrade (new version)]
  $ helm upgrade my-model ./ml-model-chart --namespace ml-prod --set image.tag=v1.3.0
  -> Updates the existing release to new values. Kubernetes does rolling update of pods.

[Install or upgrade (idempotent)]
  $ helm upgrade --install my-model ./ml-model-chart --namespace ml-prod --set image.tag=v1.3.0
  -> Safe to run repeatedly. Used in CI/CD: installs if not present, upgrades if present.

[Upgrade with values file]
  $ helm upgrade my-model ./ml-model-chart -f values-prod.yaml --set image.tag=v1.3.0
  -> Use a values file for stable config, --set for per-deploy overrides like image tag.

[Rollback to previous revision]
  $ helm rollback my-model 1
  -> Rolls back to revision 1. Helm keeps full release history. Revision 0 = previous.

[View release hi

## Section 4: GitOps Principles

GitOps is a set of practices that use Git as the single source of truth for infrastructure and application configuration. It was formalized by Weaveworks in 2017 and is now widely adopted for Kubernetes deployments.

### The Four Core Principles

**1. Declarative**: The entire system is described as a desired state, not as a sequence of imperative commands. You do not write "scale to 5 replicas" (imperative). You write "replica count: 5" (declarative). The system figures out how to get there.

**2. Versioned and Immutable**: The desired state is stored in Git. Every change is a commit with an author, timestamp, message, and diff. Rolling back means reverting a commit. There is a complete audit trail of who changed what and when.

**3. Pulled Automatically**: An agent running inside the cluster watches the Git repository and pulls changes. You do not push to the cluster directly. This eliminates the need for CI systems to have cluster credentials and reduces the attack surface.

**4. Continuously Reconciled**: The agent constantly compares the actual cluster state to the desired state in Git and corrects any drift. If someone manually changes a Kubernetes resource with `kubectl`, the GitOps agent will detect the drift and revert it to match Git.

In [7]:
# GitOps vs Traditional Deployment

traditional = {
    'Deployment trigger': 'CI pipeline runs kubectl apply or helm upgrade directly',
    'Credentials': 'CI server holds Kubernetes cluster credentials',
    'Rollback': 'Manual: re-run previous pipeline or kubectl rollout undo',
    'Audit trail': 'CI pipeline logs (often ephemeral)',
    'Drift detection': 'None: manual changes persist silently',
    'Source of truth': 'Cluster state (may diverge from any config file)'
}

gitops = {
    'Deployment trigger': 'Git commit merged to main branch',
    'Credentials': 'GitOps agent in cluster pulls from Git (no outbound creds needed)',
    'Rollback': 'git revert <commit> -> push -> auto-deployed within seconds',
    'Audit trail': 'Git log: who changed what, when, why (PR description)',
    'Drift detection': 'Agent continuously reconciles; alerts on drift',
    'Source of truth': 'Git repository (cluster always converges to match)'
}

print(f'{"Aspect":<25} {"Traditional CI/CD":<40} {"GitOps":<40}')
print('-' * 105)
for key in traditional:
    print(f'{key:<25} {traditional[key]:<40} {gitops[key]:<40}')

Aspect                    Traditional CI/CD                        GitOps                                  
---------------------------------------------------------------------------------------------------------
Deployment trigger        CI pipeline runs kubectl apply or helm upgrade directly Git commit merged to main branch        
Credentials               CI server holds Kubernetes cluster credentials GitOps agent in cluster pulls from Git (no outbound creds needed)
Rollback                  Manual: re-run previous pipeline or kubectl rollout undo git revert <commit> -> push -> auto-deployed within seconds
Audit trail               CI pipeline logs (often ephemeral)       Git log: who changed what, when, why (PR description)
Drift detection           None: manual changes persist silently    Agent continuously reconciles; alerts on drift
Source of truth           Cluster state (may diverge from any config file) Git repository (cluster always converges to match)


## Section 5: ArgoCD

ArgoCD is the most widely used GitOps operator for Kubernetes. It runs as a set of pods in your cluster and continuously watches one or more Git repositories.

### How ArgoCD Works

1. You create an **Application** resource in Kubernetes (an ArgoCD CRD)
2. The Application points to a Git repo, a path within that repo, and a target cluster/namespace
3. ArgoCD periodically polls the Git repo (or uses webhooks for instant detection)
4. When Git state differs from cluster state, ArgoCD shows the app as **OutOfSync**
5. On auto-sync, ArgoCD applies the changes. On manual sync, an operator clicks Sync in the UI

### Health Statuses

| Status | Meaning |
|---|---|
| **Healthy** | All resources are running and passing health checks |
| **Progressing** | Deployment rolling update in progress |
| **Degraded** | One or more resources failed (CrashLoopBackOff, ImagePullError) |
| **Suspended** | Auto-sync is paused |
| **Unknown** | Cannot determine health (custom resources without health check) |

In [8]:
# ArgoCD Application CRD manifest
# This YAML is applied to the cluster once; ArgoCD handles everything after that
argocd_application = """
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: fraud-detection-prod
  namespace: argocd          # ArgoCD always lives in the 'argocd' namespace
spec:
  project: default

  source:
    repoURL: https://github.com/company/ml-deployments.git
    targetRevision: main     # Watch the main branch
    path: apps/fraud-detection/prod    # Only watch this subdirectory

    # Helm-specific configuration
    helm:
      valueFiles:
        - values.yaml
        - values-prod.yaml
      parameters:
        - name: image.tag
          value: v2.1.0      # Can also be updated programmatically by CI

  destination:
    server: https://kubernetes.default.svc   # The cluster ArgoCD runs in
    namespace: ml-prod

  syncPolicy:
    automated:
      prune: true      # Delete resources removed from Git
      selfHeal: true   # Revert manual changes that diverge from Git
    syncOptions:
      - CreateNamespace=true    # Create namespace if missing
      - PrunePropagationPolicy=foreground

  # Keep last 10 revisions for rollback
  revisionHistoryLimit: 10
"""

print('ArgoCD Application manifest:')
print(argocd_application)

ArgoCD Application manifest:

apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: fraud-detection-prod
  namespace: argocd          # ArgoCD always lives in the 'argocd' namespace
spec:
  project: default

  source:
    repoURL: https://github.com/company/ml-deployments.git
    targetRevision: main     # Watch the main branch
    path: apps/fraud-detection/prod    # Only watch this subdirectory

    # Helm-specific configuration
    helm:
      valueFiles:
        - values.yaml
        - values-prod.yaml
      parameters:
        - name: image.tag
          value: v2.1.0      # Can also be updated programmatically by CI

  destination:
    server: https://kubernetes.default.svc   # The cluster ArgoCD runs in
    namespace: ml-prod

  syncPolicy:
    automated:
      prune: true      # Delete resources removed from Git
      selfHeal: true   # Revert manual changes that diverge from Git
    syncOptions:
      - CreateNamespace=true    # Create namespace if missing
     

In [9]:
# ArgoCD sync strategies
sync_strategies = {
    'Manual sync': {
        'description': 'ArgoCD shows OutOfSync but does not apply. Human clicks Sync or runs argocd app sync.',
        'use_case': 'Production environments where a human must approve each deployment',
        'yaml': 'syncPolicy: {}  # empty = manual'
    },
    'Auto-sync (no selfHeal)': {
        'description': 'Applies Git changes automatically. Manual kubectl changes persist until next Git change.',
        'use_case': 'Staging: auto-deploy from Git but allow emergency kubectl patches',
        'yaml': 'syncPolicy:\n  automated:\n    prune: true\n    selfHeal: false'
    },
    'Auto-sync + selfHeal': {
        'description': 'Applies Git changes AND reverts any manual changes. Cluster always matches Git exactly.',
        'use_case': 'Strict production: no manual changes allowed, full GitOps enforcement',
        'yaml': 'syncPolicy:\n  automated:\n    prune: true\n    selfHeal: true'
    }
}

for strategy, info in sync_strategies.items():
    print(f'\n--- {strategy} ---')
    print(f'Description: {info["description"]}')
    print(f'Use case: {info["use_case"]}')
    print(f'YAML:\n{info["yaml"]}')


--- Manual sync ---
Description: ArgoCD shows OutOfSync but does not apply. Human clicks Sync or runs argocd app sync.
Use case: Production environments where a human must approve each deployment
YAML:
syncPolicy: {}  # empty = manual

--- Auto-sync (no selfHeal) ---
Description: Applies Git changes automatically. Manual kubectl changes persist until next Git change.
Use case: Staging: auto-deploy from Git but allow emergency kubectl patches
YAML:
syncPolicy:
  automated:
    prune: true
    selfHeal: false

--- Auto-sync + selfHeal ---
Description: Applies Git changes AND reverts any manual changes. Cluster always matches Git exactly.
Use case: Strict production: no manual changes allowed, full GitOps enforcement
YAML:
syncPolicy:
  automated:
    prune: true
    selfHeal: true


## Section 6: ML Deployment with GitOps

For ML teams the typical GitOps workflow looks like this:

1. Data scientist trains a new model version, pushes Docker image to registry
2. CI pipeline updates the image tag in the Git deployment repository
3. PR is created for team review (optional in dev, required in prod)
4. PR is merged, ArgoCD detects the change within 3 minutes (or instantly via webhook)
5. ArgoCD syncs, Kubernetes does a rolling update (zero downtime)
6. ArgoCD shows Healthy; monitoring shows no metric regression
7. If metrics regress: `git revert` -> push -> ArgoCD auto-rolls back in minutes

In [10]:
# GitHub Actions workflow that builds, pushes, and updates the deployment repo
# This is the CI half; ArgoCD is the CD half
github_workflow = """
# .github/workflows/deploy.yml
# Triggered when a model training job succeeds and pushes a new image

name: Build and Deploy ML Model

on:
  push:
    branches: [main]
    paths:
      - 'src/**'
      - 'Dockerfile'

env:
  REGISTRY: myregistry.azurecr.io
  IMAGE_NAME: fraud-detection

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout model code
        uses: actions/checkout@v4

      - name: Login to container registry
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ secrets.REGISTRY_USERNAME }}
          password: ${{ secrets.REGISTRY_PASSWORD }}

      - name: Build and push image
        uses: docker/build-push-action@v5
        with:
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest

      # Update the deployment repo with the new image tag
      # ArgoCD watches this repo and will auto-deploy the change
      - name: Update image tag in deployment repo
        uses: actions/checkout@v4
        with:
          repository: company/ml-deployments
          token: ${{ secrets.DEPLOY_REPO_TOKEN }}
          path: ml-deployments

      - name: Bump image tag
        run: |
          cd ml-deployments
          # Update image tag in Helm values file
          sed -i 's|tag:.*|tag: "${{ github.sha }}"|' apps/fraud-detection/dev/values.yaml
          git config user.email ci@company.com
          git config user.name "CI Bot"
          git add apps/fraud-detection/dev/values.yaml
          git commit -m "deploy: fraud-detection to ${{ github.sha }}"
          git push
        # ArgoCD detects this commit and auto-deploys to dev
        # Promotion to staging/prod requires a separate PR
"""

print('GitHub Actions workflow (.github/workflows/deploy.yml):')
print(github_workflow)

GitHub Actions workflow (.github/workflows/deploy.yml):

# .github/workflows/deploy.yml
# Triggered when a model training job succeeds and pushes a new image

name: Build and Deploy ML Model

on:
  push:
    branches: [main]
    paths:
      - 'src/**'
      - 'Dockerfile'

env:
  REGISTRY: myregistry.azurecr.io
  IMAGE_NAME: fraud-detection

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout model code
        uses: actions/checkout@v4

      - name: Login to container registry
        uses: docker/login-action@v3
        with:
          registry: ${{ env.REGISTRY }}
          username: ${{ secrets.REGISTRY_USERNAME }}
          password: ${{ secrets.REGISTRY_PASSWORD }}

      - name: Build and push image
        uses: docker/build-push-action@v5
        with:
          push: true
          tags: |
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:${{ github.sha }}
            ${{ env.REGISTRY }}/${{ env.IMAGE_NAME }}:latest

      # Update th

In [11]:
# Environment promotion using directory structure
# This is the most common GitOps pattern for ML teams
repo_structure = """
ml-deployments/
|-- apps/
    |-- fraud-detection/
        |-- base/
        |   |-- Chart.yaml          # Chart metadata (shared)
        |   |-- templates/          # Kubernetes templates (shared)
        |   |-- values.yaml         # Default values (shared)
        |
        |-- dev/
        |   |-- values.yaml         # Dev overrides: image.tag=<git-sha>, replicas=1
        |   |-- argocd-app.yaml     # ArgoCD Application for dev cluster
        |
        |-- staging/
        |   |-- values.yaml         # Staging overrides: replicas=2, alerts enabled
        |   |-- argocd-app.yaml     # ArgoCD Application for staging cluster
        |
        |-- prod/
            |-- values.yaml         # Prod overrides: replicas=5, strict resource limits
            |-- argocd-app.yaml     # ArgoCD Application for prod cluster (manual sync)

Promotion workflow:
  1. CI auto-commits new image tag to dev/values.yaml
  2. ArgoCD auto-deploys to dev cluster
  3. Tests pass -> engineer opens PR: copy image tag from dev/values to staging/values
  4. PR reviewed, merged -> ArgoCD auto-deploys to staging
  5. Staging validation passes -> PR: staging tag to prod/values
  6. PR reviewed (may require 2 approvals) -> merged -> ArgoCD syncs prod (manual or auto)
"""

print('Deployment repository structure for multi-environment GitOps:')
print(repo_structure)

Deployment repository structure for multi-environment GitOps:

ml-deployments/
|-- apps/
    |-- fraud-detection/
        |-- base/
        |   |-- Chart.yaml          # Chart metadata (shared)
        |   |-- templates/          # Kubernetes templates (shared)
        |   |-- values.yaml         # Default values (shared)
        |
        |-- dev/
        |   |-- values.yaml         # Dev overrides: image.tag=<git-sha>, replicas=1
        |   |-- argocd-app.yaml     # ArgoCD Application for dev cluster
        |
        |-- staging/
        |   |-- values.yaml         # Staging overrides: replicas=2, alerts enabled
        |   |-- argocd-app.yaml     # ArgoCD Application for staging cluster
        |
        |-- prod/
            |-- values.yaml         # Prod overrides: replicas=5, strict resource limits
            |-- argocd-app.yaml     # ArgoCD Application for prod cluster (manual sync)

Promotion workflow:
  1. CI auto-commits new image tag to dev/values.yaml
  2. ArgoCD auto-de

## Section 7: Flux as Alternative

Flux is the other major GitOps operator, developed by Weaveworks (who coined the term GitOps) and now a CNCF Graduated project alongside ArgoCD.

**Similarities**: Both watch Git repositories, both reconcile cluster state to match, both support Helm charts, both are CNCF projects.

**Key differences**:

| Aspect | ArgoCD | Flux |
|---|---|---|
| UI | Rich web UI with visualization | Minimal UI (CLI-first) |
| Philosophy | Application-centric | Infrastructure-centric |
| Multi-tenancy | Projects and RBAC built-in | Namespaced controllers |
| OCI support | Charts from OCI registries | Strong OCI + Cosign signing |
| Config approach | Application CRD | Multiple CRDs (GitRepository, Kustomization, HelmRelease) |
| Learning curve | Lower (UI helps) | Steeper (more explicit) |

**When to choose Flux**: Teams that prefer CLI tools over dashboards, organizations that want stricter separation of concerns, or teams already invested in Kustomize-based workflows.

**When to choose ArgoCD**: Teams that need a visible dashboard for operations, multi-tenant clusters where different teams deploy different apps, or teams coming from a Helm-first background.

In [12]:
# Flux HelmRelease CRD (equivalent to ArgoCD Application for Helm deployments)
flux_helm_release = """
# Flux equivalent of the ArgoCD Application above
---
apiVersion: source.toolkit.fluxcd.io/v1
kind: GitRepository
metadata:
  name: ml-deployments
  namespace: flux-system
spec:
  interval: 1m                 # Poll Git every minute
  url: https://github.com/company/ml-deployments
  ref:
    branch: main
---
apiVersion: helm.toolkit.fluxcd.io/v2
kind: HelmRelease
metadata:
  name: fraud-detection
  namespace: ml-prod
spec:
  interval: 5m
  chart:
    spec:
      chart: apps/fraud-detection/base    # Path to Helm chart in Git repo
      sourceRef:
        kind: GitRepository
        name: ml-deployments
        namespace: flux-system
  valuesFrom:
    - kind: ConfigMap
      name: fraud-detection-prod-values  # Values stored as ConfigMap in cluster
  values:
    image:
      tag: v2.1.0
"""

print('Flux HelmRelease (alternative to ArgoCD Application):')
print(flux_helm_release)

Flux HelmRelease (alternative to ArgoCD Application):

# Flux equivalent of the ArgoCD Application above
---
apiVersion: source.toolkit.fluxcd.io/v1
kind: GitRepository
metadata:
  name: ml-deployments
  namespace: flux-system
spec:
  interval: 1m                 # Poll Git every minute
  url: https://github.com/company/ml-deployments
  ref:
    branch: main
---
apiVersion: helm.toolkit.fluxcd.io/v2
kind: HelmRelease
metadata:
  name: fraud-detection
  namespace: ml-prod
spec:
  interval: 5m
  chart:
    spec:
      chart: apps/fraud-detection/base    # Path to Helm chart in Git repo
      sourceRef:
        kind: GitRepository
        name: ml-deployments
        namespace: flux-system
  valuesFrom:
    - kind: ConfigMap
      name: fraud-detection-prod-values  # Values stored as ConfigMap in cluster
  values:
    image:
      tag: v2.1.0



## Key Takeaways

**Helm eliminates YAML duplication.** Instead of maintaining separate manifests per environment, you maintain one chart with templates and separate values files. A single `helm upgrade --install` command handles both first deploy and updates.

**GitOps makes deployments auditable and reversible.** Every change is a Git commit. Rollback is `git revert`. The audit trail is built-in. No one bypasses review by running `kubectl apply` directly in production.

**ArgoCD is the most common implementation.** It provides a web UI, drift detection, and sync strategies from fully automated to fully manual. The Application CRD is simple to configure and the dashboard makes deployment state immediately visible.

**The standard ML GitOps pattern**: CI builds and pushes image, updates image tag in deployment repo, ArgoCD detects the commit and deploys. Promotion from dev to staging to prod is a PR that copies the validated image tag to the next environment's values file.

**Separate your code repo from your deployment repo.** Model code, training scripts, and app code live in one repo. Helm charts, values files, and ArgoCD manifests live in a deployment repo. CI writes to the deployment repo; GitOps operators read from it.